In [30]:
import neuroglancer
import numpy as np

Create a new (initially empty) viewer.  This starts a webserver in a background thread, which serves a copy of the Neuroglancer client, and which also can serve local volume data and handles sending and receiving Neuroglancer state updates.

In [31]:
viewer = neuroglancer.Viewer()

Print a link to the viewer (only valid while the notebook kernel is running). Note that while the Viewer is running, anyone with the link can obtain any authentication credentials that the neuroglancer Python module obtains. Therefore, be very careful about sharing the link, and keep in mind that sharing the notebook will likely also share viewer links.

In [32]:
viewer

http://127.0.0.1:61321/v/c2337dae97f211439d354ee4f656ef14299a96ae/

Add some example layers using the precomputed data source (HHMI Janelia FlyEM FIB-25 dataset).

In [8]:
with viewer.txn() as s:
  s.layers['image'] = neuroglancer.ImageLayer(source='precomputed://gs://neuroglancer-public-data/flyem_fib-25/image')
  s.layers['segmentation'] = neuroglancer.SegmentationLayer(source='precomputed://gs://neuroglancer-public-data/flyem_fib-25/ground_truth', selected_alpha=0.3)


Display a numpy array as an additional layer.  A reference to the numpy array is kept only as long as the layer remains in the viewer.

Move the viewer position.

In [9]:
with viewer.txn() as s:
    s.voxel_coordinates = [3000.5, 3000.5, 3000.5]

Hide the segmentation layer.

In [10]:
with viewer.txn() as s:
    s.layers['segmentation'].visible = False

In [12]:
import tensorstore as ts

image_vol = await ts.open({'driver': 'neuroglancer_precomputed', 'kvstore': 'gs://neuroglancer-public-data/flyem_fib-25/image/'})
a = np.zeros((200,200,200), np.uint8)
def make_thresholded(threshold):
  a[...] = image_vol[3000:3200,3000:3200,3000:3200][...,0].read().result() > threshold
make_thresholded(110)
# This volume handle can be used to notify the viewer that the data has changed.
volume = neuroglancer.LocalVolume(
    a,
    dimensions=neuroglancer.CoordinateSpace(
        names=['x', 'y', 'z'],
        units='nm',
        scales=[8, 8, 8],
    ),
    voxel_offset=[3000, 3000, 3000])
with viewer.txn() as s:
  s.layers['overlay'] = neuroglancer.ImageLayer(
        source=volume,
      # Define a custom shader to display this mask array as red+alpha.
        shader="""
void main() {
  float v = toNormalized(getDataValue(0)) * 255.0;
  emitRGBA(vec4(v, 0.0, 0.0, v));
}
""",
    )

I0000 00:00:1721413034.884764 1163423 gcs_resource.cc:109] Using default AdmissionQueue with limit 32
W0000 00:00:1721413034.931758 1165743 curl_transport.cc:452] Error [6]=Couldn't resolve host name in curl operation
Could not resolve host: metadata.google.internal
E0000 00:00:1721413034.931958 1165748 google_auth_provider.cc:187] Could not find the credentials file in the standard gcloud location [/Users/wanqing.yu/.config/gcloud/application_default_credentials.json]. You may specify a credentials file using $GOOGLE_APPLICATION_CREDENTIALS, or to use Google application default credentials, run: gcloud auth application-default login


Modify the overlay volume, and call `invalidate()` to notify the Neuroglancer client.

In [16]:
make_thresholded(100)
volume.invalidate()

Select a couple segments.

In [19]:
with viewer.txn() as s:
    # s.layers['segmentation'].segments.update([1752, 88847])
    s.layers['segmentation'].visible = True

Print the neuroglancer viewer state.  The Neuroglancer Python library provides a set of Python objects that wrap the JSON-encoded viewer state.  `viewer.state` returns a read-only snapshot of the state.  To modify the state, use the `viewer.txn()` function, or `viewer.set_state`.

In [20]:
viewer.state

ViewerState({"dimensions": {"x": [8e-09, "m"], "y": [8e-09, "m"], "z": [8e-09, "m"]}, "position": [3000.5, 3000.5, 3000.5], "crossSectionScale": 1, "projectionScale": 8192, "layers": [{"type": "image", "source": "precomputed://gs://neuroglancer-public-data/flyem_fib-25/image", "tab": "source", "name": "image"}, {"type": "segmentation", "source": "precomputed://gs://neuroglancer-public-data/flyem_fib-25/ground_truth", "tab": "source", "selectedAlpha": 0.3, "segments": [], "name": "segmentation"}, {"type": "image", "source": "python://volume/55aa1723cfe439383d221a75030121a1ccb83dbf.0f9f3af8e1dc47435b4736b9e82975e689cf9fc7", "tab": "source", "shader": "\nvoid main() {\n  float v = toNormalized(getDataValue(0)) * 255.0;\n  emitRGBA(vec4(v, 0.0, 0.0, v));\n}\n", "name": "overlay"}], "layout": "4panel"})

Print the set of selected segments.|

In [21]:
viewer.state.layers['segmentation'].segments

VisibleSegments([])

Update the state by calling `set_state` directly.

In [22]:
import copy

new_state = copy.deepcopy(viewer.state)
new_state.layers['segmentation'].segments.add(10625)
viewer.set_state(new_state)

'a98cd10c9f7440319adff385fc20269b391d1e6c'

Bind the 't' key in neuroglancer to a Python action.

In [23]:
num_actions = 0
def my_action(s):
    global num_actions
    num_actions += 1
    with viewer.config_state.txn() as st:
      st.status_messages['hello'] = ('Got action %d: mouse position = %r' %
                                     (num_actions, s.mouse_voxel_coordinates))
    print('Got my-action')
    print(f'  Mouse position: {s.mouse_voxel_coordinates}')
    print(f'  Layer selected values: {s.selected_values}')
viewer.actions.add('my-action', my_action)
with viewer.config_state.txn() as s:
    s.input_event_bindings.viewer['keyt'] = 'my-action'
    s.status_messages['hello'] = 'Welcome to this example'

Change the view layout to 3-d.

In [24]:
with viewer.txn() as s:
    s.layout = '3d'
    s.projection_scale = 3000

Take a screenshot (useful for creating publication figures, or for generating videos).  While capturing the screenshot, we hide the UI and specify the viewer size so that we get a result independent of the browser size.

In [25]:
from ipywidgets import Image

screenshot = viewer.screenshot(size=[1000, 1000])
screenshot_image = Image(value=screenshot.screenshot.image)
screenshot_image

Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x03\xe8\x00\x00\x03\xe8\x08\x06\x00\x00\x00M\xa3\xd4…

Change the view layout to show the segmentation side by side with the image, rather than overlayed.  This can also be done from the UI by dragging and dropping.  The side by side views by default have synchronized position, orientation, and zoom level, but this can be changed.

In [26]:
with viewer.txn() as s:
    s.layout = neuroglancer.row_layout(
        [neuroglancer.LayerGroupViewer(layers=['image', 'overlay']),
         neuroglancer.LayerGroupViewer(layers=['segmentation'])])

Remove the overlay layer.

In [27]:
with viewer.txn() as s:
    s.layout = neuroglancer.row_layout(
        [neuroglancer.LayerGroupViewer(layers=['image']),
         neuroglancer.LayerGroupViewer(layers=['segmentation'])])

Create a publicly sharable URL to the viewer state (only works for external data sources, not layers served from Python).  The Python objects for representing the viewer state (`neuroglancer.ViewerState` and friends) can also be used independently from the interactive Python-tied viewer to create Neuroglancer links.

In [28]:
print(neuroglancer.to_url(viewer.state))

https://neuroglancer-demo.appspot.com#!%7B%22dimensions%22:%7B%22x%22:%5B8e-09,%22m%22%5D,%22y%22:%5B8e-09,%22m%22%5D,%22z%22:%5B8e-09,%22m%22%5D%7D,%22position%22:%5B3000.5,3000.5,3000.5%5D,%22crossSectionScale%22:1,%22projectionScale%22:3000,%22layers%22:%5B%7B%22type%22:%22image%22,%22source%22:%22precomputed://gs://neuroglancer-public-data/flyem_fib-25/image%22,%22tab%22:%22source%22,%22name%22:%22image%22%7D,%7B%22type%22:%22segmentation%22,%22source%22:%22precomputed://gs://neuroglancer-public-data/flyem_fib-25/ground_truth%22,%22tab%22:%22source%22,%22selectedAlpha%22:0.3,%22segments%22:%5B%2210625%22%5D,%22name%22:%22segmentation%22%7D%5D,%22layout%22:%7B%22type%22:%22row%22,%22children%22:%5B%7B%22type%22:%22viewer%22,%22layers%22:%5B%22image%22%5D,%22layout%22:%22xy%22%7D,%7B%22type%22:%22viewer%22,%22layers%22:%5B%22segmentation%22%5D,%22layout%22:%22xy%22%7D%5D%7D%7D


Stop the Neuroglancer web server, which invalidates any existing links to the Python-tied viewer.

In [33]:
neuroglancer.stop()